In [1]:
"""
==============================================================
DATA PREPARATION v2 - Deteksi Stunting Posyandu
==============================================================
Perubahan dari v1:
  - Undersample kelas Normal → 800 (rasio 2:1)
  - Drop fitur Rasio_BB_TB (redundan dengan BMI)
  - Tambah cek distribusi sebelum & sesudah undersample
==============================================================
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv('../data/raw/bismillah_final.csv')
print(f"✅ Data berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 2: CEK KOLOM USIA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: CEK KOLOM USIA")
print("=" * 55)

print("Distribusi usia (dalam bulan) sebelum filter:")
print(df['Usia_Bulan'].value_counts().sort_index().to_string())
print()


# ──────────────────────────────────────────────
# FASE 3: FILTER HANYA BALITA (0-60 BULAN)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: FILTER BALITA (USIA ≤ 60 BULAN)")
print("=" * 55)

sebelum = len(df)
df = df[df['Usia_Bulan'] <= 60].copy()
sesudah = len(df)
print(f"✅ Data sebelum filter : {sebelum} baris")
print(f"✅ Data setelah filter : {sesudah} baris")
print(f"   Dibuang            : {sebelum - sesudah} baris (usia > 60 bulan)\n")


# ──────────────────────────────────────────────
# FASE 4: BERSIHKAN KOLOM JK
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: BERSIHKAN KOLOM JENIS KELAMIN")
print("=" * 55)

print(f"   Nilai unik sebelum : {df['JK'].unique()}")
df['JK'] = df['JK'].str.strip()
print(f"   Nilai unik sesudah : {df['JK'].unique()}\n")


# ──────────────────────────────────────────────
# FASE 5: BUAT LABEL STATUS STUNTING (dari ZS TB/U)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: BUAT LABEL STATUS STUNTING")
print("=" * 55)

def buat_status_stunting(zs_tbu):
    """
    Klasifikasi stunting berdasarkan standar WHO:
    ZS TB/U < -3         : Sangat Pendek
    -3 <= ZS TB/U < -2   : Pendek
    ZS TB/U >= -2        : Normal
    """
    if pd.isna(zs_tbu):
        return np.nan
    elif zs_tbu < -3:
        return 'Sangat Pendek'
    elif zs_tbu < -2:
        return 'Pendek'
    else:
        return 'Normal'

df['Status_Stunting'] = df['ZS TB/U'].apply(buat_status_stunting)
print("✅ Distribusi Status Stunting (sebelum cleaning):")
for label, count in df['Status_Stunting'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%)")
print()


# ──────────────────────────────────────────────
# FASE 6: TANGANI MISSING VALUES
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: TANGANI MISSING VALUES")
print("=" * 55)

print("Missing values sebelum ditangani:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print()

sebelum = len(df)
df.dropna(subset=['Status_Stunting'], inplace=True)
sesudah = len(df)
print(f"✅ Baris dibuang karena ZS TB/U kosong: {sebelum - sesudah}")

# Drop kolom ZS setelah label dibuat
df.drop(columns=['ZS BB/U', 'ZS TB/U', 'ZS BB/TB'], inplace=True)

print(f"\nMissing values setelah ditangani:")
print(df.isnull().sum().to_string())
print()


# ──────────────────────────────────────────────
# FASE 7: TAMBAH FITUR TURUNAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: TAMBAH FITUR TURUNAN")
print("=" * 55)

# BMI = Berat (kg) / Tinggi (m)²
df['BMI'] = df['Berat'] / ((df['Tinggi'] / 100) ** 2)

# [v2] Rasio_BB_TB TIDAK ditambahkan — redundan dengan BMI
print("✅ BMI ditambahkan sebagai fitur turunan")
print("ℹ️  Rasio_BB_TB DIHAPUS — informasinya sudah tercakup dalam BMI\n")

# Cek outlier sebelum filter
print(f"Sebelum filter outlier: {len(df)} baris")
print(f"BMI    : min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")
print(f"Tinggi : min={df['Tinggi'].min():.2f}, max={df['Tinggi'].max():.2f}")
print(f"Berat  : min={df['Berat'].min():.2f}, max={df['Berat'].max():.2f}")

# Filter nilai tidak wajar
df = df[df['Tinggi'] >= 40].copy()
df = df[df['Tinggi'] <= 120].copy()
df = df[df['Berat'] >= 1].copy()
df = df[df['Berat'] <= 30].copy()
df = df[df['BMI'] <= 40].copy()

print(f"Setelah filter outlier : {len(df)} baris")
print(f"BMI    : min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")
print(f"✅ Statistik fitur turunan:")
print(f"   BMI : mean={df['BMI'].mean():.2f}, min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")
print()


# ──────────────────────────────────────────────
# FASE 8: ENCODING VARIABEL KATEGORIKAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: ENCODING VARIABEL KATEGORIKAL")
print("=" * 55)

df['JK_encoded'] = df['JK'].map({'L': 1, 'P': 0})
print("✅ JK encoded: L=1, P=0")

le_stunting = LabelEncoder()
df['Status_Stunting_encoded'] = le_stunting.fit_transform(df['Status_Stunting'])
print(f"✅ Status Stunting encoded:")
for label, kode in zip(le_stunting.classes_, le_stunting.transform(le_stunting.classes_)):
    print(f"   {label:<15} = {kode}")
print()


# ──────────────────────────────────────────────
# FASE 9: SUSUN DATASET SEBELUM UNDERSAMPLE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: SUSUN DATASET")
print("=" * 55)

# [v2] Fitur tanpa Rasio_BB_TB
feature_cols = ['JK_encoded', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI']

df_clean = df[feature_cols + ['Status_Stunting']].copy()
df_clean.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Status_Stunting']

print(f"✅ Fitur  : {['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI']}")
print(f"✅ Target : Status_Stunting")
print(f"   Kelas  : {le_stunting.classes_.tolist()}")
print(f"\nShape dataset sebelum undersample: {df_clean.shape}")
print()


# ──────────────────────────────────────────────
# FASE 10: UNDERSAMPLE KELAS NORMAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 10: UNDERSAMPLE KELAS NORMAL (n=800)")
print("=" * 55)

print("Distribusi SEBELUM undersample:")
for label, count in df_clean['Status_Stunting'].value_counts().items():
    pct = count / len(df_clean) * 100
    bar = 'X' * int(pct / 2)
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%) {bar}")
print(f"   Total: {len(df_clean)} baris\n")

# Cek jumlah Normal, sesuaikan kalau kurang dari 800
n_normal_asli = len(df_clean[df_clean['Status_Stunting'] == 'Normal'])
n_undersample  = min(800, n_normal_asli)

if n_normal_asli < 800:
    print(f"⚠️  Data Normal hanya {n_normal_asli} baris, undersample ke {n_undersample} (semua dipakai)")
else:
    print(f"✅ Undersample Normal: {n_normal_asli} → {n_undersample} baris")

df_normal        = df_clean[df_clean['Status_Stunting'] == 'Normal'].sample(n=n_undersample, random_state=42)
df_pendek        = df_clean[df_clean['Status_Stunting'] == 'Pendek']
df_sangat_pendek = df_clean[df_clean['Status_Stunting'] == 'Sangat Pendek']

df_clean = pd.concat([df_normal, df_pendek, df_sangat_pendek]) \
             .sample(frac=1, random_state=42) \
             .reset_index(drop=True)

print("\nDistribusi SESUDAH undersample:")
for label, count in df_clean['Status_Stunting'].value_counts().items():
    pct = count / len(df_clean) * 100
    bar = 'X' * int(pct / 2)
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%) {bar}")
print(f"   Total: {len(df_clean)} baris\n")

# Cek imbalance
min_pct = df_clean['Status_Stunting'].value_counts(normalize=True).min() * 100
if min_pct < 15:
    print("⚠️  Masih ada class imbalance — pastikan pakai class_weight='balanced' saat training.")
else:
    print("✅ Distribusi kelas cukup seimbang.")
print()


# ──────────────────────────────────────────────
# FASE 11: SIMPAN DATASET BERSIH
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 11: SIMPAN DATASET")
print("=" * 55)

df_clean.to_csv('../data/processed/dataset_clean_v2.csv', index=False)
print("✅ dataset_clean_v2.csv tersimpan!")
print(f"   Berisi {len(df_clean)} baris dan {len(df_clean.columns)} kolom")
print(f"   Kolom: {df_clean.columns.tolist()}")
print()


# ──────────────────────────────────────────────
# RINGKASAN AKHIR
# ──────────────────────────────────────────────
print("=" * 55)
print("✅ DATA PREPARATION v2 SELESAI")
print("=" * 55)
print(f"   Fitur yang dipakai  : JK, Usia_Bulan, Berat, Tinggi, BMI")
print(f"   Fitur dihapus       : Rasio_BB_TB (redundan)")
print(f"   Kelas Normal        : diundersample ke {n_undersample} baris")
print(f"   Total data final    : {len(df_clean)} baris")
print(f"   Target              : Status Stunting (3 kelas)")
print(f"   Mapping label       : Normal=0 | Pendek=1 | Sangat Pendek=2")
print(f"   Siap untuk training Random Forest!")
print("=" * 55)

FASE 1: LOAD DATA
✅ Data berhasil dimuat: 2881 baris, 7 kolom
   Kolom: ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS TB/U', 'ZS BB/U', 'ZS BB/TB']

FASE 2: CEK KOLOM USIA
Distribusi usia (dalam bulan) sebelum filter:
Usia_Bulan
0      10
1      34
2      21
3      20
4      20
5      23
6      37
7      29
8      25
9      29
10     30
11     17
12     24
13     28
14     25
15     18
16     15
17     25
18     25
19     29
20     21
21     19
22     22
23     31
24     85
25     25
26     21
27     24
28     36
29     41
30     25
31     30
32     17
33     22
34     32
35     42
36    233
37     38
38     39
39     20
40     28
41     15
42     38
43     37
44     28
45     27
46     33
47     39
48    340
49     32
50     31
51     28
52     29
53     37
54     19
55     26
56     26
57     22
58     16
59     15
60    331
72    232
84    133
96     12

FASE 3: FILTER BALITA (USIA ≤ 60 BULAN)
✅ Data sebelum filter : 2881 baris
✅ Data setelah filter : 2504 baris
   Dibuang            